In [ ]:
#library for ML framework 
import tensorflow as tf
"""
Sequential is the way to build model with the tf framework, we
can add layers, build, compile and fit the complete model 
"""
from tensorflow.keras.models import Sequential 
"""
Embedding layer - the bridge from events (represented as integers via IDs) and make
them vectors with fixed size - https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding
how to represent events to computers?
We can have a fixed amount of events, each one represented with vector full of zeros except one slot of the specific event 
this is called one-hot encoding the shortcoming of this method is the lack of context between similar events on the system. 
in addition, the data is very sparse but still in this method spend a lot of space.
with Embedding layer we represent an event with a vector full of real numbers (in most cases the vector has low dimensions)
during training this layer can learn when events occur in similar conditions and then the points on the vector space are become close to each other. 
In this project the dimension of each vector is 16 and we have 175 evetns (matrix 175X16)
each row is a specific event (like hashtable) this vector is inputed to the LSTM layer 

LSTM layer - Long Short Term Memory (Hochreiter 1997) represented the LSTM model that I covered in the article 
https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM

Dense Layer - this is the normal NN connection when each neuron connected to each one of the next layer
we can choose the activation function (linear on defualt)
in this case for exaple the output layer is like this and we get vacab_size outputs (one for each event) 
and output the unnormalize probability to each event to be the next 

"""
from tensorflow.keras.models import Embedding, LSTM, Dense


def build_ladohd_model(vocab_size=175, embedding_dim=16, hidden_size=64, seq_length=64):
    """
    Builds the LADOHD LSTM anomaly detection model.
    
    Args:
        vocab_size (int): Total number of unique system events(fixed number).
        embedding_dim (int): Dimension of the dense embedding vectors.
        hidden_size (int): Number of features in the LSTM hidden state.
        seq_length (int): Length of the input event sequences (BPTT window).
        
    Returns:
        tf.keras.Model: The compiled sequential model.
    """
    model = Sequential([
        # 1. Embedding Layer: Converts categorical event IDs into dense vectors
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=seq_length),
        
        # 2. LSTM Layers: 3 stacked layers to learn complex temporal patterns
        # return_sequences=True ensures the entire sequence is passed to the next layer
        LSTM(hidden_size, return_sequences=True),
        LSTM(hidden_size, return_sequences=True),
        LSTM(hidden_size, return_sequences=True),
        
        # 3. Fully Connected Layer: Extracts non-linear features from the LSTM outputs
        Dense(100, activation='relu'),
        
        # 4. Output Layer: Unnormalized log probabilities (logits) for each event in the vocabulary
        Dense(vocab_size)
    ])
    
    return model

if __name__ == "__main__":
    model = build_ladohd_model()
    
    # Compile the model with Adam optimizer and Sparse Categorical Crossentropy loss
    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    
    model.summary()
